# Week 18 · Notebook 02: Foundry Agent & Evaluation

# Requirements: pip install azure-ai-projects azure-identity openai

# ⚠️ REQUIRES: Azure (AI Foundry project)

> 💰 COST WARNING: set a budget alert before running, agent runs and evaluation batches bill per token.


## What you build

Define the ZoroLogistics support agent with a tool (function calling), run an evaluation batch on a golden set, read the trace, fix the worst failure, and print a before/after eval score. Missing credentials → dry-run mode with a deterministic local "trace" stand-in.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import numpy as np
import pandas as pd
from zoro import data

SEED = 18
rng = np.random.default_rng(SEED)

ship = data.shipments(n=5_000, seed=42)
policies = data.policy_docs()
print("Loaded", len(ship), "shipments and", len(policies), "policy docs.")


## The support agent's tool

An agent = model + instructions + tools. We define one function-calling tool, `track_shipment`, which looks up a real (synthetic) shipment id and returns status/ETA. It mirrors the Week 14 to 16 behavior.


In [ ]:
def track_shipment(tracking_number: str) -> dict:
    """Return status and ETA for a ZoroLogistics shipment id."""
    row = ship[ship["shipment_id"] == str(tracking_number).strip()]
    if row.empty:
        return {"found": False, "message": "No shipment with that tracking id."}
    s = row.iloc[0]
    return {
        "found": True,
        "shipment_id": str(s["shipment_id"]),
        "status": str(s["status"]),
        "carrier_id": str(s["carrier_id"]),
        "commodity": str(s["commodity"]),
        "is_on_time": bool(s["is_on_time"]),
    }

print("track_shipment('S0000001') ->", track_shipment("S0000001"))


## Define the agent (azure-ai-projects)

`AIProjectClient` creates the agent with a model, a name, instructions, and tools. The instructions encode the Week 14 to 16 behavior: never invent a tracking number; route refunds over $500 to human approval.


In [ ]:
AGENT_INSTRUCTIONS = (
    "You are the ZoroLogistics support agent. Help customers track shipments and answer "
    "shipping-policy questions. Use the track_shipment tool for tracking lookups. Never "
    "invent a tracking number. Flag refund requests above $500 for human approval."
)

agent = None
if os.environ.get("AZURE_AI_PROJECT_CONNECTION_STRING"):
    try:
        from azure.ai.projects import AIProjectClient
        from azure.identity import DefaultAzureCredential
        client = AIProjectClient.from_connection_string(
            conn_str=os.environ["AZURE_AI_PROJECT_CONNECTION_STRING"],
            credential=DefaultAzureCredential(),
        )
        agent = client.agents.create_agent(
            model="gpt-4.1",
            name="zoro-support-agent",
            instructions=AGENT_INSTRUCTIONS,
            tools=[],  # attach the function-calling tool here once registered
        )
        print("✅ Created agent:", agent.id)
    except Exception as e:  # noqa: BLE001
        print("⚠️ Agent creation failed:", type(e).__name__, e)
        agent = None
else:
    print("DRY-RUN: set AZURE_AI_PROJECT_CONNECTION_STRING to create the agent.")


## Golden eval set

Port the Week 11 golden set: question → expected answer. We build a tiny one from the policy docs and support tickets so the eval has ground truth to score against.


In [ ]:
golden = [
    {"id": "Q1", "question": "How late can a shipment arrive before I get a refund?",
     "expected_contains": ["48 hours", "10%"], "doc_id": "POL-002"},
    {"id": "Q2", "question": "What is the address-change fee after pickup?",
     "expected_contains": ["$85"], "doc_id": "POL-001"},
    {"id": "Q3", "question": "What documentation does a dangerous-goods shipment need?",
     "expected_contains": ["shipper's declaration", "UN number"], "doc_id": "POL-003"},
    {"id": "Q4", "question": "What is the storage fee for customs holds beyond 5 days?",
     "expected_contains": ["$40"], "doc_id": "POL-004"},
]
print("Golden set has", len(golden), "questions.")


## Run the evaluation batch

Ask each question, get an answer, and score it against the expected substrings. This is a rough proxy for a full LLM-as-judge; Foundry's built-in evaluators (groundedness, relevance) are the production version. In dry-run we use a small keyword retriever over the policy docs, deliberately imperfect, so the "fix the worst failure" step has something to fix.


In [ ]:
# Deliberately incomplete keyword index (misses 'customs' on purpose).
KEYWORD_DOCS = {"refund": "POL-002", "late": "POL-002",
               "address": "POL-001", "dangerous": "POL-003"}

def answer_question(question):
    q = question.lower()
    doc_id = next((d for k, d in KEYWORD_DOCS.items() if k in q), None)
    if doc_id:
        for d in policies:
            if d["doc_id"] == doc_id:
                return d["text"]
    return "I could not find an answer in the shipping policies."

def run_eval():
    trace, rows = [], []
    for item in golden:
        ans = answer_question(item["question"])
        hits = [k for k in item["expected_contains"] if k.lower() in ans.lower()]
        passed = len(hits) == len(item["expected_contains"])
        trace.append({"id": item["id"], "question": item["question"],
                      "answer": ans[:90], "hits": hits, "passed": passed})
        rows.append({"id": item["id"], "passed": passed, "hits": len(hits),
                     "expected": len(item["expected_contains"])})
    return pd.DataFrame(rows), trace

eval_df, trace = run_eval()
before_score = eval_df["passed"].mean() if len(eval_df) else 0.0
print(eval_df.to_string(index=False))
print(f"\nBEFORE_SCORE: {before_score:.3f}")


## Read the trace

In Foundry, traces appear in the portal and record inputs, tool calls, intermediate steps, and outputs. Here we print the local trace stand-in, the same shape of signal: which question, what the agent said, and whether it passed.


In [ ]:
trace_df = pd.DataFrame(trace)
print(trace_df[['id', 'passed', 'hits', 'answer']].to_string(index=False))
print("\nNOTE: in Foundry this is the end-to-end trace in the portal, tool calls and all.")


## Fix the worst failure

The eval above should fail Q4 (customs), because the keyword index lacks the `customs` → POL-004 mapping. The fix is to add it, then re-run. In a live agent this same loop means editing the instructions or adding a tool; here it is one mapping.


In [ ]:
worst = eval_df.sort_values("passed").iloc[0]
print("Worst failure:", worst["id"], "passed =", worst["passed"])

# THE FIX: add the missing keyword -> doc mapping.
KEYWORD_DOCS["customs"] = "POL-004"
KEYWORD_DOCS["storage"] = "POL-004"

eval_df_after, trace_after = run_eval()
after_score = eval_df_after["passed"].mean() if len(eval_df_after) else 0.0
print(eval_df_after.to_string(index=False))
print(f"\nAFTER_SCORE: {after_score:.3f}")


In [ ]:
# Final number: the before -> after improvement (eval score).
EVAL_SCORE = after_score
print(f"EVAL_SCORE: {EVAL_SCORE:.3f}")
print(f"improvement: {before_score:.3f} -> {after_score:.3f}")
